**Helper functions for analyzing multi-value columns.**

`explode_multi_value` / `value_counts_multi` turn a comma-separated column into one-item-per-row for frequency counts.  
`contains_value` builds a boolean mask so it can filter/count perfumes by any note, accord, or perfumer. 
With an `exact` flag to choose between a whole-item match (`"Lemon"`) and a substring match (`"lemon"` also catching `"Italian lemon"`).

In [ ]:
def explode_multi_value(df, column, sep=","):
    """One row per individual item in a comma-separated column (whitespace stripped)."""
    return df[column].str.split(sep).explode().str.strip()


def value_counts_multi(df, column, sep=","):
    """Frequency count of individual values inside a comma-separated column."""
    return explode_multi_value(df, column, sep).value_counts()


def contains_value(df, columns, term, exact=True, case_sensitive=False):
    """
    Boolean mask over df: rows where any of `columns` contains `term`.

    exact=True  -> matches a whole comma-separated item ("Lemon" != "Italian lemon")
    exact=False -> substring match ("Lemon" also matches "Italian lemon zest")
    """
    if isinstance(columns, str):
        columns = [columns]

    term_cmp = term if case_sensitive else term.lower()
    mask = pd.Series(False, index=df.index)

    def row_match(cell_items):
        if not isinstance(cell_items, list):
            return False
        for item in cell_items:
            item = item.strip()
            item_cmp = item if case_sensitive else item.lower()
            if (exact and item_cmp == term_cmp) or (not exact and term_cmp in item_cmp):
                return True
        return False

    for col in columns:
        mask = mask | df[col].str.split(",").apply(row_match)

    return mask

In [ ]:
value_counts_multi(df, "Perfumers").head(10)   # most prolific perfumers
value_counts_multi(df, "Main_Accords")          # frequency of every accord

In [ ]:
# average rating per Top Note
notes = explode_multi_value(df, "Top_Notes")          # Series: index=original row, value=note
notes_df = notes.to_frame("Top_Note").join(df[["Rating_Value", "Brand"]])
notes_df.groupby("Top_Note")["Rating_Value"].mean().sort_values(ascending=False).head(10)

In [ ]:
NOTE_COLUMNS = ["Top_Notes", "Middle_Notes", "Base_Notes"]

# Exact-token match: "Lemon" only matches the standalone note "Lemon"
lemon_exact = contains_value(df, NOTE_COLUMNS, "Lemon", exact=True)
print(f"Perfumes with 'Lemon' as an exact note: {lemon_exact.sum()}")

# Substring match: also catches variants like "Italian lemon", "lemon zest", "Amalfi lemon"
lemon_any = contains_value(df, NOTE_COLUMNS, "lemon", exact=False)
print(f"Perfumes with any note containing 'lemon': {lemon_any.sum()}")

# General analysis: most common Top Notes overall
value_counts_multi(df, "Top_Notes").head(10)